# Find what needs review in a robot dataset

Grade [`lerobot/aloha_sim_transfer_cube_human`](https://huggingface.co/datasets/lerobot/aloha_sim_transfer_cube_human), 50 simulated ALOHA episodes, with Kalanos and see what it flags.
CPU only, no API key, nothing is published. Choose **Runtime → Run all**.

## Install

Kalanos and two helpers, at pinned versions.

In [ ]:
# @title Install Kalanos
%pip install -q --no-warn-conflicts \
    "kalanos[numeric,hf]==0.10.0" huggingface-hub==1.33.0 plotly==7.1.0

## Download the dataset

About 70 MB, pinned to one revision so every run sees the same files.

In [ ]:
# @title Download the dataset
from pathlib import Path

from huggingface_hub import snapshot_download


DATASET_ID = "lerobot/aloha_sim_transfer_cube_human"
# A fixed revision keeps the files, and so every result below, the same on each run.
REVISION = "6a43d500f101255823a9d2b9dc244eeb01a2cd31"

DATASET = Path(
    snapshot_download(
        repo_id=DATASET_ID,
        repo_type="dataset",
        revision=REVISION,
        local_dir="aloha_sim_transfer_cube_human",
    )
)

## Grade it

The CLI prints a report card and saves the full report as JSON: 11 of 50 episodes need review.

In [ ]:
# @title Grade it with the Kalanos CLI
# `--verbosity error` hides warnings about file formats this dataset doesn't use.
# Piping through `cat` stops the CLI from animating a progress spinner in the cell,
# and `--color` keeps the card's colours through the pipe.
!kalanos --verbosity error grade {DATASET} --report audit.json --color 2>&1 | cat

## Explore the report in Python

One row per episode. The score and the decision are separate: episode 22 scores lowest but passes, because only a check past its review bound sends an episode to review ([how decisions work](https://github.com/KalanosAI/kalanos/blob/v0.10.0/docs/DECISIONS.md)).

In [ ]:
# @title Load the report into a table
import polars as pl

import kalanos


loaded = kalanos.load_report("audit.json")
# `load_report` also reads reports from older schemas, which have no episodes.
# Assigning after the check gives `report` that narrower type in later cells too.
assert isinstance(loaded, kalanos.Report)
report = loaded

rows = [
    {"episode": e.id, "decision": e.eligibility.status.value, "score": e.score.score}
    for e in report.episodes
    # Every episode in a saved report has a decision; this only tells the type checker.
    if e.eligibility is not None
]

# Wide enough to show the episode number at the end of each id.
pl.Config.set_fmt_str_lengths(50)
pl.DataFrame(rows).with_columns(pl.col("score").round(1)).sort("score")

## What Kalanos flagged

Kalanos ran 14 checks on this dataset ([all metrics](https://github.com/KalanosAI/kalanos/blob/v0.10.0/docs/METRICS.md)); signal-to-noise on the left-waist command is the one that sends episodes to review.
In the flagged episodes the command barely moves, so its flicker outweighs its motion. Kalanos calls this a heuristic: a reason to look, not proof of a fault.

Camera checks need `kalanos[video]` and a [vision profile](https://github.com/KalanosAI/kalanos/blob/v0.10.0/docs/PROFILES.md#vision-requirements).

In [ ]:
# @title Chart what Kalanos flagged
import json
import math

import plotly.express as px


CHANNEL = "left_waist"
# Pass and review differ in hue but barely in lightness, so review also gets a diamond.
PASS_COLOR, REVIEW_COLOR, REVIEW_TINT = "#23735e", "#87511b", "#f8eddc"
BLOCKED_COLOR, UNKNOWN_COLOR = "#a62b23", "#65625c"
PAPER, SURFACE, RULE, INK, MUTED = "#f4f1ea", "#ffffff", "#d2cec5", "#171717", "#65625c"
MONO = "'IBM Plex Mono', monospace"

# From the dataset: how far the command moves from where each episode starts,
# counted in its smallest step, since the recorded values sit on a fixed grid.
info = json.loads((DATASET / "meta" / "info.json").read_text())
position = info["features"]["action"]["names"]["motors"].index(CHANNEL)
command = pl.col("action").list.get(position)
change = (
    pl.read_parquet(DATASET / "data" / "**" / "*.parquet")
    .sort("episode_index", "frame_index")
    .select("episode_index", change=(command - command.first()).over("episode_index"))
)
step = change.filter(pl.col("change") != 0)["change"].abs().min()
moves = change.group_by("episode_index").agg(
    steps=(pl.col("change").abs().max() / step).round().cast(pl.Int64)
)

# From Kalanos: the channel's signal-to-noise ratio and the episode's decision,
# found in the report as episode → action stream → channel → metric.
verdicts = []
for e in report.episodes:
    stream = next(s for s in e.streams if s.source_field == "action")
    channel = next(c for c in stream.channels if c.channel.name == CHANNEL)
    snr_db = channel.metrics["snr_db"].value
    # A channel that only switches between two values has no ratio.
    if snr_db is not None:
        decision = e.eligibility.status.value if e.eligibility else "unknown"
        verdicts.append(
            {
                "episode_index": int(e.id.rsplit("_", 1)[1]),
                "snr_db": snr_db,
                "decision": decision,
            }
        )
episodes = pl.DataFrame(verdicts).join(moves, on="episode_index")
policy = kalanos.load_policy(None)
review_below = policy.metrics["integrity.snr_db"].thresholds["default"].bad

fig = px.scatter(
    episodes,
    x="steps",
    y="snr_db",
    color="decision",
    color_discrete_map={
        "pass": PASS_COLOR,
        "review": REVIEW_COLOR,
        "blocked": BLOCKED_COLOR,
        "unknown": UNKNOWN_COLOR,
    },
    symbol="decision",
    symbol_map={
        "pass": "circle",
        "review": "diamond",
        "blocked": "x",
        "unknown": "square",
    },
    category_orders={"decision": ["review", "pass"]},
    custom_data=["episode_index", "decision"],
    log_x=True,
    # A bold tag on each axis title says where its numbers come from.
    labels={
        "steps": f'<span style="font-family:{MONO}"><b>DATASET</b></span> · how far {CHANNEL} moves, in steps of {step:.4f}',
        "snr_db": f'<span style="font-family:{MONO}"><b>KALANOS</b></span> · signal-to-noise (dB)',
    },
    height=600,
)
# A paper-coloured ring keeps overlapping dots apart.
fig.update_traces(
    marker={"size": 15, "line": {"color": PAPER, "width": 2}},
    hovertemplate="<b>Episode %{customdata[0]}</b><br>moves %{x} steps"
    "<br>%{y:.1f} dB · %{customdata[1]}<extra></extra>",
)

# Shade the range Kalanos sends to review, with its bound drawn behind the dots.
fig.add_hrect(
    y0=min(v["snr_db"] for v in verdicts) - 5,
    y1=review_below,
    fillcolor=REVIEW_TINT,
    line_width=0,
    layer="below",
    annotation_text=f"Kalanos sends episodes below {review_below:g} dB to review",
    annotation_font_color=MUTED,
)
fig.add_hline(y=review_below, line_color=REVIEW_COLOR, line_width=1.5, layer="below")

# Point at the episode with the lowest ratio, and at the passing episode
# that moves least, which shows the split isn't clean.
lowest = episodes.sort("snr_db").row(0, named=True)
closest_pass = (
    episodes.filter(pl.col("decision") == "pass")
    .sort("steps", "snr_db")
    .row(0, named=True)
)
for e, text, ax, ay in [
    (lowest, "barely moves", 330, 0),
    (closest_pass, "moves about as little, but passes", -200, -20),
]:
    fig.add_annotation(
        x=math.log10(e["steps"]),
        y=e["snr_db"],
        text=f"<b>Episode {e['episode_index']}</b> {text}<br>"
        f"{e['steps']} {'step' if e['steps'] == 1 else 'steps'}"
        f" · {e['snr_db']:.0f} dB · {e['decision']}",
        ax=ax,
        ay=ay,
        align="left",
        standoff=9,
        bgcolor=SURFACE,
        bordercolor=RULE,
        borderpad=8,
        arrowcolor=MUTED,
    )
axis = {"gridcolor": RULE, "linecolor": RULE, "zeroline": False, "ticks": ""}
fig.update_xaxes(axis, tickvals=[1, 2, 4, 8, 16, 32, 64], title_standoff=16)
fig.update_yaxes(axis, title_standoff=12)
# Counted over every episode, so one with no ratio still counts towards the title.
n_review = sum(
    e.eligibility is not None and e.eligibility.status.value == "review"
    for e in report.episodes
)
fig.update_layout(
    font={
        "family": "'IBM Plex Sans', Arial, sans-serif",
        "size": 15,
        "color": INK,
    },
    paper_bgcolor=PAPER,
    plot_bgcolor=PAPER,
    margin={"t": 130, "r": 30},
    title={
        "text": f"The {n_review} of {len(report.episodes)} episodes Kalanos sends to review"
        f" barely move the {CHANNEL} command",
        "font": {"size": 22},
        "subtitle": {
            "text": "One dot per episode. Little movement is a sign, not a rule.<br>"
            f"{len(report.episodes) - len(verdicts)} episodes have no ratio:"
            " their command only switches between two values.",
            "font": {"color": MUTED, "size": 15},
        },
    },
    legend={"title_text": "Kalanos decision", "x": 0.01, "y": 0.98},
)
fig.show()

## Download the reports

The report as an HTML page and as JSON, zipped.

In [ ]:
# @title Write the HTML report
# The same grade again, written as an HTML page.
# `tail` keeps the card's last lines: the file written, or the error if it failed.
!kalanos --verbosity error grade {DATASET} --report audit.html 2>&1 | tail -n 2

In [ ]:
# @title Download both
import zipfile

from IPython.display import FileLink, display


with zipfile.ZipFile("kalanos-reports.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write("audit.json")
    archive.write("audit.html")

try:
    # Only Colab has this module; it sends the file to the browser.
    from google.colab import files  # pyright: ignore[reportMissingImports]
except ImportError:
    display(FileLink("kalanos-reports.zip"))
else:
    files.download("kalanos-reports.zip")

## Continue with Kalanos

- [View the dataset](https://huggingface.co/datasets/lerobot/aloha_sim_transfer_cube_human): the public LeRobot dataset graded here.
- [Browse public dataset reports](https://hub.kalanos.ai): investigate existing assessments; their revisions and scopes may differ from this run.
- [Kalanos website](https://kalanos.ai): learn about the toolkit and workflows.
- [Grader source and methodology](https://github.com/KalanosAI/kalanos): run local audits and inspect decision semantics.
- [MCP integrations](https://github.com/KalanosAI/kalanos-mcp): consume published evidence in agent workflows; not required for this notebook.
- [Experiment tracking adapters](https://github.com/KalanosAI/kalanos-tracking): attach existing assessments to experiments; sharing is a separate choice.
- [Synthetic fixture generator](https://hub.kalanos.ai/synthetic-data): sign-in required.